#  07. ANFIS Training (MLP Surrogate)

##  Objective: Train MLP surrogate model on 6-feature ANFIS inputs.

## ℹ️ Note on Option B (Why we chose this):
The initial ANFIS attempts failed to generalize (R² < 0) due to "variance collapse" the target variable had insufficient spread for learning (σ = 0.0113).
**Option B** redesigned the target function to use behavioral deltas (changes over time) as primary variance drivers. This increased target variance by 5.5x and restored learnability, achieving **R² ≈ 0.96** on unseen data. This is the **Canonical v2.2** implementation required for the game integration.

##  Output Artifacts: Trained model parameters 
### (saved to `../../data/models/anfis_mlp_weights.json`)


In [ ]:
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import json
import os

INPUT_PATH = '../../data/processed/6_anfis_dataset.csv'
MODEL_DIR = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries imported")

In [ ]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

In [ ]:
# Train/Test Split
feature_cols = [
    'soft_combat', 'soft_collect', 'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

X = df[feature_cols].values
y = df['target_multiplier'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training with {len(feature_cols)} features")

## Architecture Decision: MLP Surrogate

> **Why not a raw Fuzzy Inference System (FIS)?**
>
> 1. **Runtime Speed:** In a 60 FPS game loop, evaluating hundreds of fuzzy rules is slower than a simple Matrix Multiplication (MLP).
> 2. **Portability:** Game Engines (Unity/Unreal) handle Linear Algebra natively. Custom Fuzzy parsers are fragile.
>
> **The Approach:** We used Fuzzy Logic to *design the features* (Soft Membership), but used an MLP to *approximate the surface*. Best of both worlds.



In [ ]:
# Train MLP
print("Training MLP...")
model = MLPRegressor(hidden_layer_sizes=(16, 8), activation='relu', max_iter=500, random_state=42)
model.fit(X_train, y_train)
print("Training complete")

In [ ]:
# Save params
train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))

params = {
    'architecture': '6-16-8-1',
    'metrics': {'train_mae': train_mae, 'test_mae': test_mae},
    'features': feature_cols
}

with open(os.path.join(MODEL_DIR, 'anfis_params.json'), 'w') as f:
    json.dump(params, f, indent=2)
    
print(f"Model saved. Test MAE: {test_mae:.4f}")

In [ ]:
# FINAL EXPORT - CANONICAL (v2.2)
import json, numpy as np
from sklearn.metrics import r2_score, mean_squared_error

y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

train_r2  = r2_score(y_train, y_train_pred)
test_r2   = r2_score(y_test,  y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse  = mean_squared_error(y_test,  y_test_pred)

# ── Neutral baseline ────────────────────────────────────────────────────────
# MLP output for a perfectly balanced, zero-delta player.
# The live engine maps this -> display 1.0 (no difficulty change).
# Values above neutral -> harder; below neutral -> easier.
neutral_input = np.array([[1/3, 1/3, 1/3, 0, 0, 0]])
mlp_neutral = float(model.predict(neutral_input)[0])

print('='*60)
print('TRAINING COMPLETE - OPTION B CANONICAL')
print('='*60)
print(f'Train R²:    {train_r2:.4f}')
print(f'Test  R²:    {test_r2:.4f}')
print(f'Train MAE:   {train_mae:.6f}')
print(f'Test  MAE:   {test_mae:.6f}')
print(f'MLP neutral: {mlp_neutral:.6f}  (balanced, zero-delta input)')
print('='*60)

export_data = {
    'architecture': {
        'input_size': 6,
        'hidden_layers': [16, 8],
        'output_size': 1,
        'activation': 'relu',
        'output_activation': 'linear'
    },
    'feature_order': feature_cols,
    'weights': [w.tolist() for w in model.coefs_],
    'biases':  [b.tolist() for b in model.intercepts_],
    'training_metrics': {
        'train_mae':      float(train_mae),
        'test_mae':       float(test_mae),
        'train_mse':      float(train_mse),
        'test_mse':       float(test_mse),
        'train_r2':       float(train_r2),
        'test_r2':        float(test_r2),
        'num_iterations': model.n_iter_,
        'num_samples':    len(X)
    },
    # Neutral-centred calibration ──────────────────────────────────────────
    # The live engine applies:  display = clamp(1.0 + (raw - mlp_neutral) * 2.0, 0.6, 1.4)
    # mlp_neutral is recomputed here after every retrain so no JS code ever needs updating.
    'mlp_neutral': round(mlp_neutral, 6),
    'version': 'v2.2-optionB-canonical',
    'status':  'ACTIVE'
}

save_path = '../../data/models/anfis_mlp_weights.json'
with open(save_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f'[done] Weights exported -> {save_path}')


##  Model Benchmarking & Ablation Study

To validate the system's performance, we compare the current ANFIS-derived MLP against baseline models (Linear Regression, Random Forest) and perform an ablation study on the 'delta' features to justify their inclusion in the canonical pipeline.

In [ ]:
# ── Fix 9.3: No-delta ablation (clean feature_sets iteration) ──────────────
# Demonstrates that temporal deltas meaningfully improve prediction over
# soft-membership alone, justifying the added pipeline complexity.
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

feature_sets = {
    'soft_only': ['soft_combat', 'soft_collect', 'soft_explore'],
    'soft_plus_delta': [
        'soft_combat', 'soft_collect', 'soft_explore',
        'delta_combat', 'delta_collect', 'delta_explore'
    ],
}

ablation_results = []

for name, cols in feature_sets.items():
    X_abl = df[cols].values
    y_abl = df['target_multiplier'].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_abl, y_abl, test_size=0.2, random_state=42
    )

    mlp_abl = MLPRegressor(
        hidden_layer_sizes=(16, 8),
        activation='relu',
        solver='lbfgs',
        max_iter=500,
        random_state=42
    )
    mlp_abl.fit(X_tr, y_tr)
    y_pred_abl = mlp_abl.predict(X_te)

    ablation_results.append({
        'feature_set': name,
        'n_features': len(cols),
        'r2': r2_score(y_te, y_pred_abl),
        'mae': mean_absolute_error(y_te, y_pred_abl),
    })

ablation_df = pd.DataFrame(ablation_results).sort_values('r2', ascending=False)
print('No-Delta Ablation Results:')
print(ablation_df.to_string(index=False))

delta_r2  = ablation_df.loc[ablation_df['feature_set'] == 'soft_plus_delta', 'r2'].values[0]
nodelta_r2 = ablation_df.loc[ablation_df['feature_set'] == 'soft_only', 'r2'].values[0]
print(f'Delta contribution: R² improvement = {delta_r2 - nodelta_r2:+.4f}')


In [ ]:
# ── Deploy model artifacts -> anfis-demo-ui/models/ ──────────────────
import shutil, os
DEPLOY_DIR = r'f:\\Campus\\FYP\\Implementation\\CollectGame.Model\\anfis-demo-ui\\models'
os.makedirs(DEPLOY_DIR, exist_ok=True)
FILES_TO_DEPLOY = [
    'anfis_mlp_weights.json',
    'scaler_params.json',
    'deployment_manifest.json',
]
for fname in FILES_TO_DEPLOY:
    src_file = os.path.join(MODEL_DIR, fname)
    if os.path.exists(src_file):
        shutil.copy2(src_file, DEPLOY_DIR)
        print(f'  Deployed {fname} -> anfis-demo-ui/models/')
    else:
        print(f'  WARN: {fname} not found in data/models/')
print('Deploy complete.')


In [ ]:
# ── Deploy artifacts + save training_stats.json ──────────────────────────
import shutil, json, numpy as np

MODEL_DIR   = '../../data/models'
DEPLOY_DIR  = r'f:\Campus\FYP\Implementation\CollectGame.Model\anfis-demo-ui\models'
os.makedirs(DEPLOY_DIR, exist_ok=True)

# 1. Deploy core model artifacts
FILES_TO_DEPLOY = ['anfis_mlp_weights.json', 'scaler_params.json', 'deployment_manifest.json']
for fname in FILES_TO_DEPLOY:
    src = os.path.join(MODEL_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, DEPLOY_DIR)
        print(f'  [done] {fname} -> anfis-demo-ui/models/')
    else:
        print(f'  WARN: {fname} not found')

# 2. Save training_stats.json (target distribution + membership ranges)
t = df['target_multiplier']
stats = {
    'num_samples': int(len(df)),
    'target_multiplier': {
        'min':  float(t.min()),
        'max':  float(t.max()),
        'mean': float(t.mean()),
        'std':  float(t.std())
    },
    'soft_membership_ranges': {
        col: [float(df[col].min()), float(df[col].max())]
        for col in ['soft_combat', 'soft_collect', 'soft_explore']
    },
    'mlp_neutral': round(mlp_neutral, 6)
}
stats_path = os.path.join(DEPLOY_DIR, 'training_stats.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'  [done] training_stats.json -> anfis-demo-ui/models/')

print('\nTarget multiplier distribution:')
print(f'  min={t.min():.4f}  max={t.max():.4f}  mean={t.mean():.4f}  std={t.std():.4f}')
print(f'  mlp_neutral={mlp_neutral:.6f}')
print('\nDeploy complete.')
